In [1]:
import pandas as pd
import numpy as np
import os
import sys

In [2]:
table_path = '/media/mad3/Projects/ICU_SLEEP_STUDY/Undiagnosed_Apnea/OSA Paper Patient Data.xlsx'

save_table_path1 = '/home/wolfgang/repos/Undiagnosed_Apnea/icu_sleep_analysis_table.csv'
save_table_path2 =  '/media/mad3/Projects/ICU_SLEEP_STUDY/Undiagnosed_Apnea/icu_sleep_analysis_table.csv'

# table = pd.read_csv(table_path)
table = pd.read_excel(table_path)
table = table.dropna(subset=['MRN'])
table = table[table['Study ID'] != '98a']
table.loc[table['Study ID']==31, 'MRN'] = 1876849
table.loc[table['Study ID']==15, 'MRN'] = 1479357
table.loc[table['Study ID']==15, 'MRN'] = 1479357


In [3]:
table.shape

(188, 18)

In [4]:
table['Study ID'].iloc[95:100]

95      96
96      97
97      98
99      99
100    100
Name: Study ID, dtype: object

In [5]:
# CHARLSON COMORBIDITY INDEX:

cci_path = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/CCI/CCI_edw_icd_codes_icu_sleep_2020_07_07.csv'
cci = pd.read_csv(cci_path)

In [6]:
table.head(3)

,Study ID,MRN,First day enrolled,Age (yrs) during ICU stay,BMI (kg/m^2) during ICU stay,"Gender (0 = female, 1 = male)",Previous OSA diagnosis (w notes),"Previous OSA diagnosis (1=yes, 0=no)",ICU mortality,"3 month mortaltiy (1= yes, 0=no)",ICU LOS (days),Hospital LOS (days),ICU readmission during hospital encounter,Primary diagnosis,LVEF (%),Date of Echo,Hx/o CHF,h/o COPD
0,1,1298742.0,2018-06-06,79.0,25.82,1.0,no,0.0,0.0,0.0,3.0,5.0,no,acute lower GI hemorrhage,76,2018-03-09 00:00:00,no,yes
1,2,6378271.0,2018-06-11,22.0,19.29,1.0,no,0.0,0.0,0.0,3.0,3.0,no,MVC,61,2018-06-09 00:00:00,no,no
2,3,1910753.0,2018-06-18,84.0,29.32,1.0,no,0.0,0.0,0.0,3.0,10.0,no,sepsis,54,2018-06-18 00:00:00,yes,yes


## ADD CCI.

In [7]:
for loc, row in table.iterrows():
    mrn_tmp = row['MRN']
    
    cci_value = cci[cci.PatientIdentityID == mrn_tmp].score
    wcci_value = cci[cci.PatientIdentityID == mrn_tmp].wscore
    if cci_value.shape[0] == 0:
        print(f'No CCI value for {row.MRN}, {row["Study ID"]}. Use 0.')
        cci_value = pd.Series([0])
        wcci_value = pd.Series([0])
    table.loc[loc, 'CCI'] = cci_value.values[0]
    table.loc[loc, 'CCI_weighted'] = wcci_value.values[0]

No CCI value for 6563939.0, 43. Use 0.


### ADD BELT LENGTH


In [8]:
belt_data = pd.read_excel('/media/mad3/Projects/ICU_SLEEP_STUDY/Undiagnosed_Apnea/ICU Airgo Belt Lengths.xlsx')
belt_data.columns = [x.strip() for x in belt_data.columns]

for loc, row in table.iterrows():
    
    studyid_tmp = row['Study ID']
    belt_length = belt_data[belt_data.ID == int(studyid_tmp)]['Belt Length (cm)']

    table.loc[loc, 'Belt_Length'] = belt_length.values[0]

In [9]:
table

,Study ID,MRN,First day enrolled,Age (yrs) during ICU stay,BMI (kg/m^2) during ICU stay,"Gender (0 = female, 1 = male)",Previous OSA diagnosis (w notes),"Previous OSA diagnosis (1=yes, 0=no)",ICU mortality,"3 month mortaltiy (1= yes, 0=no)",...,Hospital LOS (days),ICU readmission during hospital encounter,Primary diagnosis,LVEF (%),Date of Echo,Hx/o CHF,h/o COPD,CCI,CCI_weighted,Belt_Length
0,1,1298742.0,2018-06-06,79.0,25.82,1.0,no,0.0,0.0,0.0,...,5.0,no,acute lower GI hemorrhage,76,2018-03-09 00:00:00,no,yes,3.0,3.0,NaN
1,2,6378271.0,2018-06-11,22.0,19.29,1.0,no,0.0,0.0,0.0,...,3.0,no,MVC,61,2018-06-09 00:00:00,no,no,0.0,0.0,NaN
2,3,1910753.0,2018-06-18,84.0,29.32,1.0,no,0.0,0.0,0.0,...,10.0,no,sepsis,54,2018-06-18 00:00:00,yes,yes,6.0,12.0,NaN
3,4,6188100.0,2018-06-19,32.0,23.78,0.0,no,0.0,0.0,0.0,...,22.0,no,sepsis,60,2018-08-02 00:00:00,no,no,0.0,0.0,NaN
4,5,6051800.0,2018-06-19,72.0,35.52,0.0,yes,1.0,0.0,1.0,...,15.0,no,abdominal abscess,66,2018-04-17 00:00:00,yes,yes,5.0,6.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184,185,1933710.0,2019-11-13,67.0,23.99,0.0,no,0.0,0.0,NaN,...,9.0,no,septic shock,76,2019-10-11 00:00:00,no,no,1.0,2.0,84.5
185,186,6807388.0,2019-11-12,79.0,32.25,0.0,no,0.0,0.0,NaN,...,11.0,no ?,aortic dissection,71,2019-11-13 00:00:00,no,no,1.0,1.0,90.0
186,187,1849492.0,2019-11-12,71.0,23.29,1.0,yes,1.0,0.0,NaN,...,80.0,yes,"AMS, respiratory failure",65,2019-11-19 00:00:00,no,yes,0.0,0.0,71.0
187,188,5407879.0,2019-11-13,62.0,24.51,0.0,no,0.0,0.0,NaN,...,8.0,no,s/p lumbar fusion,none,-,no,no,0.0,0.0,83.0


In [10]:
# SAVE TABLE.
table.to_csv(save_table_path1, index=False)
table.to_csv(save_table_path1, index=False)